In [1]:
import os, numpy as np, pandas as pd, json

# ---- 1. What's in the downloaded model/feature cache? ----
MODELS = "/kaggle/input/datasets/ajfaisal002/misleading-models"
print("=== files in misleading-models ===")
for f in sorted(os.listdir(MODELS)):
    p = os.path.join(MODELS, f)
    print(f"  {f}  ({os.path.getsize(p)//1024} KB)")

# ---- 2. What arrays are inside each npz? ----
for name in ["feats_train.npz", "feats_test.npz", "ft_train.npz", "ft_test.npz", "ft_feats_partial.npz"]:
    p = os.path.join(MODELS, name)
    if os.path.exists(p):
        d = np.load(p, allow_pickle=True)
        print(f"\n=== {name} ===")
        for k in d.files:
            arr = d[k]
            print(f"  {k}: shape={arr.shape}, dtype={arr.dtype}")
        # show label distribution if present
        if "lab" in d.files:
            print("  lab distribution:", np.bincount(d['lab'].astype(int)).tolist())
        if "vids" in d.files:
            print("  sample vids:", d['vids'][:3].tolist())
    else:
        print(f"\n{name}: NOT FOUND")

# ---- 3. Does the Crossmodal metadata have subcategory labels? ----
CM = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text"
for split in ["train", "test"]:
    csv = os.path.join(CM, f"{split}_metadata.csv")
    df = pd.read_csv(csv)
    print(f"\n=== {split}_metadata.csv ===")
    print("  columns:", list(df.columns))
    print("  subcategory values:", df['subcategory'].value_counts(dropna=False).to_dict())

# ---- 4. Peek at the saved model checkpoints (what keys / architecture?) ----
import torch
for m in ["best_stage1_model.pth", "best_stage2_model.pth"]:
    p = os.path.join(MODELS, m)
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            print(f"\n=== {m} ===")
            if isinstance(ck, dict):
                keys = list(ck.keys())
                print("  top-level keys:", keys[:10])
                # if it's a state_dict, show a few layer names
                sd = ck.get('state_dict', ck)
                if isinstance(sd, dict):
                    print("  sample layer names:", list(sd.keys())[:8])
        except Exception as e:
            print(f"  could not load {m}: {e}")

=== files in misleading-models ===
  best_stage1_model.pth  (3852 KB)
  best_stage2_model.pth  (3856 KB)
  feats_test.npz  (31084 KB)
  feats_train.npz  (62051 KB)
  ft_feats_partial.npz  (77184 KB)
  ft_test.npz  (15553 KB)
  ft_train.npz  (61719 KB)

=== feats_train.npz ===
  vis: shape=(1600, 16, 512), dtype=float32
  aud: shape=(1600, 768), dtype=float32
  txl: shape=(1600, 768), dtype=float32
  tqw: shape=(1600, 1024), dtype=float32
  lab: shape=(1600,), dtype=int64
  vids: shape=(1600,), dtype=<U8
  lab distribution: [800, 800]
  sample vids: ['IF_001', 'IF_002', 'IF_003']

=== feats_test.npz ===
  vis: shape=(800, 16, 512), dtype=float32
  aud: shape=(800, 768), dtype=float32
  txl: shape=(800, 768), dtype=float32
  tqw: shape=(800, 1024), dtype=float32
  lab: shape=(800,), dtype=int64
  vids: shape=(800,), dtype=<U13
  lab distribution: [400, 400]
  sample vids: ['TEST_IF_001', 'TEST_IF_002', 'TEST_IF_003']

=== ft_train.npz ===
  vis: shape=(1593, 16, 512), dtype=float32
  aud

In [2]:
import os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, confusion_matrix)
from sklearn.model_selection import StratifiedKFold

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE)

MODELS = "/kaggle/input/datasets/ajfaisal002/misleading-models"
CM_TXT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text"

DIMS = {"vis":512, "aud":768, "txl":768, "tqw":1024}
SUBMAP = {"identity_fabrication":0, "perception_manipulation":1,
          "scientifically_unrealistic_scene":2, "surreal_content":3}
SUBNAMES = ["IF","PM","SUS","SC"]

# ---- subcategory lookup from metadata ----
def sub_lookup(split):
    df = pd.read_csv(os.path.join(CM_TXT, f"{split}_metadata.csv"))
    m = {}
    for _, r in df.iterrows():
        sc = r["subcategory"]
        m[str(r["video_id"])] = SUBMAP.get(sc, -1)  # -1 = safe (no subcategory)
    return m
sub_tr, sub_te = sub_lookup("train"), sub_lookup("test")

def load_cm(split, submap):
    d = np.load(os.path.join(MODELS, f"feats_{split}.npz"), allow_pickle=True)
    vids = [str(v) for v in d["vids"]]
    sub = np.array([submap.get(v, -1) for v in vids], dtype=np.int64)
    return {
        "vis_seq": torch.tensor(d["vis"]),
        "vis": torch.tensor(d["vis"].mean(1)),
        "aud": torch.tensor(d["aud"]),
        "txl": torch.tensor(d["txl"]),
        "tqw": torch.tensor(d["tqw"]),
        "lab": torch.tensor(d["lab"]).long(),          # binary: 0=safe,1=misleading
        "sub": torch.tensor(sub).long(),               # 0..3 for misleading, -1 for safe
    }

def load_ft(split):
    d = np.load(os.path.join(MODELS, f"ft_{split}.npz"), allow_pickle=True)
    return {
        "vis_seq": torch.tensor(d["vis"]),
        "vis": torch.tensor(d["vis"].mean(1)),
        "aud": torch.tensor(d["aud"]),
        "txl": torch.tensor(d["txl"]),
        "tqw": torch.tensor(d["tqw"]),
        "lab": torch.tensor(d["lab"]).long(),
    }

CM_TR, CM_TE = load_cm("train", sub_tr), load_cm("test", sub_te)
FT_TR, FT_TE = load_ft("train"), load_ft("test")

# sanity: misleading count should equal sum of subcategory counts
print("CM train binary:", torch.bincount(CM_TR["lab"]).tolist())
print("CM train sub (misleading only):", torch.bincount(CM_TR["sub"][CM_TR["sub"]>=0]).tolist())
print("CM test  sub (misleading only):", torch.bincount(CM_TE["sub"][CM_TE["sub"]>=0]).tolist())
print("FT train binary:", torch.bincount(FT_TR["lab"]).tolist(),
      "| FT test:", torch.bincount(FT_TE["lab"]).tolist())

Device: cuda:0
CM train binary: [800, 800]
CM train sub (misleading only): [200, 200, 200, 200]
CM test  sub (misleading only): [100, 100, 100, 100]
FT train binary: [655, 938] | FT test: [164, 235]


In [3]:
class ModalityProj(nn.Module):
    def __init__(self, in_dim, out_dim=384, p=0.4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(p))
    def forward(self, x): return self.net(x)

class TemporalAttn(nn.Module):
    def __init__(self, dim=512):
        super().__init__(); self.w = nn.Linear(dim, 1)
    def forward(self, seq):
        a = torch.softmax(self.w(seq).squeeze(-1), dim=1)
        return (a.unsqueeze(-1) * seq).sum(1)

class CrossAttn(nn.Module):
    def __init__(self, dim=384, heads=4):
        super().__init__(); self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)
    def forward(self, q, kv):
        q, kv = q.unsqueeze(1), kv.unsqueeze(1)
        o, _ = self.mha(q, kv, kv); return o.squeeze(1)

class FusionModel(nn.Module):
    """Proposed CA-GF-AMW fusion. n_classes=2 for binary, 4 for subcategory."""
    def __init__(self, mods=["vis","aud","txl","tqw"], d=384, dropout=0.4, n_classes=2, use_temporal=True):
        super().__init__()
        self.mods, self.d, self.use_temporal = mods, d, use_temporal
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d, dropout) for m in mods})
        M = len(mods)
        self.cross = CrossAttn(d)
        self.gate = nn.Sequential(nn.Linear(d*M, d*M), nn.Sigmoid())
        self.wgt = nn.Linear(d*M, M)
        self.head = nn.Sequential(nn.Linear(d, 128), nn.LayerNorm(128), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_classes))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            if m == "vis" and self.use_temporal:
                feats[m] = self.proj[m](self.tattn(batch["vis_seq"]))
            else:
                feats[m] = self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch)
        mats = [f[m] for m in self.mods]
        M = len(mats)
        cat = torch.cat(mats, dim=-1)
        others = sum(mats[1:]) / max(M-1, 1)
        fcross = self.cross(mats[0], others) + mats[0]
        fgated = (self.gate(cat) * cat).view(cat.size(0), M, self.d).mean(1)
        w = torch.softmax(self.wgt(cat), dim=-1)
        fadapt = sum(w[:, i:i+1] * mats[i] for i in range(M))
        fused = fcross + fgated + fadapt
        return self.head(fused)

def move(batch, idx=None):
    out = {}
    for k, v in batch.items():
        out[k] = (v[idx] if idx is not None else v).to(DEVICE)
    return out

def compute_metrics(y_true, y_pred, y_prob=None, n_classes=2):
    m = {
        "Accuracy": accuracy_score(y_true, y_pred)*100,
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0)*100,
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0)*100,
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)*100,
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)*100,
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    if n_classes == 2 and y_prob is not None:
        m["ROC-AUC"] = roc_auc_score(y_true, y_prob)
    elif y_prob is not None:
        try:
            m["ROC-AUC"] = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
        except Exception:
            m["ROC-AUC"] = float("nan")
    return m

def fmt(d):
    return {k: (round(v,2) if k not in ("ROC-AUC","MCC") else round(v,3)) for k,v in d.items()}

def train_eval(mods, train, evalset, n_classes=2, label_key="lab",
               d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60, bs=64,
               use_temporal=True, seed=SEED, filter_fn=None):
    """filter_fn: optional function(batch)->mask to subset samples (used for Stage-2 misleading-only)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = FusionModel(mods, d=d, dropout=dropout, n_classes=n_classes, use_temporal=use_temporal).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss()

    # optionally filter training set (e.g. Stage-2: misleading-only)
    if filter_fn is not None:
        mask = filter_fn(train)
        train = {k: v[mask] for k, v in train.items()}
    N = train[label_key].shape[0]; idx_all = np.arange(N)

    for ep in range(epochs):
        model.train(); np.random.shuffle(idx_all)
        for s in range(0, N, bs):
            bidx = idx_all[s:s+bs]
            b = move(train, bidx)
            logits = model(b)
            loss = lossf(logits, b[label_key])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()

    eval_set = evalset
    if filter_fn is not None:
        mask = filter_fn(evalset)
        eval_set = {k: v[mask] for k, v in evalset.items()}

    model.eval()
    with torch.no_grad():
        b = move(eval_set)
        logits = model(b)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        pred = logits.argmax(1).cpu().numpy()
        true = eval_set[label_key].numpy()
    prob_for_auc = prob[:,1] if n_classes == 2 else prob
    return compute_metrics(true, pred, prob_for_auc, n_classes), model, (true, pred, prob)

print("Cell 2 ready.")

Cell 2 ready.


In [4]:
print("=== Crossmodal STAGE 1: Binary (Safe vs Misleading) ===\n")
m1, model1, _ = train_eval(["vis","aud","txl","tqw"], CM_TR, CM_TE,
                           n_classes=2, label_key="lab", d=384, dropout=0.4,
                           lr=2e-3, wd=1e-2, epochs=60)
for k, v in fmt(m1).items():
    print(f"  {k:12s}: {v}")

=== Crossmodal STAGE 1: Binary (Safe vs Misleading) ===

  Accuracy    : 87.38
  Precision   : 87.38
  Recall      : 87.38
  Macro F1    : 87.37
  Weighted F1 : 87.37
  MCC         : 0.748
  ROC-AUC     : 0.939


In [5]:
def misleading_mask(batch):
    return batch["sub"] >= 0

print("=== Crossmodal STAGE 2: Subcategory (IF / PM / SUS / SC) ===\n")
m2, model2, (true2, pred2, prob2) = train_eval(
    ["vis","aud","txl","tqw"], CM_TR, CM_TE,
    n_classes=4, label_key="sub", d=384, dropout=0.4,
    lr=2e-3, wd=1e-2, epochs=60, filter_fn=misleading_mask
)
for k, v in fmt(m2).items():
    print(f"  {k:12s}: {v}")

# per-class breakdown
from sklearn.metrics import classification_report
print("\nPer-class report:")
print(classification_report(true2, pred2, target_names=SUBNAMES, digits=3))

cm2 = confusion_matrix(true2, pred2)
print("\nConfusion matrix (rows=true, cols=pred), order IF,PM,SUS,SC:")
print(cm2)

=== Crossmodal STAGE 2: Subcategory (IF / PM / SUS / SC) ===

  Accuracy    : 81.75
  Precision   : 83.74
  Recall      : 81.75
  Macro F1    : 81.95
  Weighted F1 : 81.95
  MCC         : 0.761
  ROC-AUC     : 0.957

Per-class report:
              precision    recall  f1-score   support

          IF      0.865     0.770     0.815       100
          PM      0.700     0.840     0.764       100
         SUS      0.784     0.910     0.843       100
          SC      1.000     0.750     0.857       100

    accuracy                          0.818       400
   macro avg      0.837     0.818     0.820       400
weighted avg      0.837     0.818     0.820       400


Confusion matrix (rows=true, cols=pred), order IF,PM,SUS,SC:
[[77 21  2  0]
 [11 84  5  0]
 [ 1  8 91  0]
 [ 0  7 18 75]]


In [6]:
print("=== Hierarchical End-to-End Evaluation ===\n")
# Stage 1 predictions on the FULL test set (both safe and misleading)
model1.eval()
with torch.no_grad():
    b1 = move(CM_TE)
    logits1 = model1(b1)
    pred1 = logits1.argmax(1).cpu().numpy()   # 0=safe, 1=misleading
true1 = CM_TE["lab"].numpy()
true_sub = CM_TE["sub"].numpy()               # -1 for safe, 0-3 for misleading

# Stage 2 predictions ONLY on samples Stage 1 flagged as misleading
model2.eval()
final_pred = np.full(len(true1), -1)  # -1 = predicted safe
misleading_idx = np.where(pred1 == 1)[0]
if len(misleading_idx) > 0:
    sub_batch = {k: v[misleading_idx] for k, v in CM_TE.items()}
    with torch.no_grad():
        b2 = move(sub_batch)
        logits2 = model2(b2)
        pred2_cascade = logits2.argmax(1).cpu().numpy()
    final_pred[misleading_idx] = pred2_cascade

# Build a unified 5-class ground truth / prediction: 0=Safe,1=IF,2=PM,3=SUS,4=SC
def to_unified(binary_lab, sub_lab):
    out = np.zeros(len(binary_lab), dtype=int)
    for i in range(len(binary_lab)):
        out[i] = 0 if binary_lab[i] == 0 else sub_lab[i] + 1
    return out

true_unified = to_unified(true1, true_sub)
pred_unified = np.where(final_pred == -1, 0, final_pred + 1)  # cascade: safe if Stage1 said safe

unified_names = ["Safe","IF","PM","SUS","SC"]
print("Hierarchical (cascade) 5-way classification report:")
print(classification_report(true_unified, pred_unified, target_names=unified_names, digits=3, zero_division=0))

hier_acc = accuracy_score(true_unified, pred_unified) * 100
hier_f1 = f1_score(true_unified, pred_unified, average="macro", zero_division=0) * 100
print(f"\nOverall hierarchical accuracy: {hier_acc:.2f}%")
print(f"Overall hierarchical Macro F1: {hier_f1:.2f}%")
print(f"\nStage-1 samples misrouted (Safe predicted Misleading, or vice versa): "
      f"{(pred1 != true1).sum()} / {len(true1)}")

=== Hierarchical End-to-End Evaluation ===

Hierarchical (cascade) 5-way classification report:
              precision    recall  f1-score   support

        Safe      0.869     0.880     0.875       400
          IF      0.703     0.520     0.598       100
          PM      0.682     0.730     0.705       100
         SUS      0.645     0.890     0.748       100
          SC      0.961     0.730     0.830       100

    accuracy                          0.799       800
   macro avg      0.772     0.750     0.751       800
weighted avg      0.808     0.799     0.797       800


Overall hierarchical accuracy: 79.88%
Overall hierarchical Macro F1: 75.10%

Stage-1 samples misrouted (Safe predicted Misleading, or vice versa): 101 / 800


# cell-2 fusion part modified

In [8]:
class FusionModel(nn.Module):
    """Proposed CA-GF-AMW fusion. n_classes=2 for binary, 4 for subcategory."""
    def __init__(self, mods=["vis","aud","txl","tqw"], d=384, dropout=0.4, n_classes=2, use_temporal=True):
        super().__init__()
        self.mods, self.d, self.use_temporal = mods, d, use_temporal
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d, dropout) for m in mods})
        M = len(mods)
        self.cross = CrossAttn(d)
        self.gate = nn.Sequential(nn.Linear(d*M, d*M), nn.Sigmoid())
        self.wgt = nn.Linear(d*M, M)
        self.head = nn.Sequential(nn.Linear(d, 128), nn.LayerNorm(128), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_classes))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            if m == "vis" and self.use_temporal:
                feats[m] = self.proj[m](self.tattn(batch["vis_seq"]))
            else:
                feats[m] = self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch)
        mats = [f[m] for m in self.mods]
        M = len(mats)
        cat = torch.cat(mats, dim=-1)

        if M >= 2:
            others = sum(mats[1:]) / (M - 1)
            fcross = self.cross(mats[0], others) + mats[0]
        else:
            fcross = mats[0]  # nothing to cross-attend to with a single modality

        fgated = (self.gate(cat) * cat).view(cat.size(0), M, self.d).mean(1)
        w = torch.softmax(self.wgt(cat), dim=-1)
        fadapt = sum(w[:, i:i+1] * mats[i] for i in range(M))
        fused = fcross + fgated + fadapt
        return self.head(fused)

print("FusionModel patched.")

FusionModel patched.


In [9]:
def run_stage2(mods, **kw):
    m, _, _ = train_eval(mods, CM_TR, CM_TE, n_classes=4, label_key="sub",
                         d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60,
                         filter_fn=misleading_mask, use_temporal=("vis" in mods), **kw)
    return fmt(m)

print("=== Stage 2: Unimodal ===\n")
uni = {"CLIP":["vis"], "Wav2Vec2":["aud"], "XLM-R":["txl"], "Qwen":["tqw"]}
for name, mods in uni.items():
    r = run_stage2(mods)
    print(f"  {name:10s} Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  MCC={r['MCC']:.3f}")

print("\n=== Stage 2: Bimodal / Trimodal ===\n")
combos = {
    "CLIP+Wav2Vec2": ["vis","aud"],
    "CLIP+XLM-R+Qwen": ["vis","txl","tqw"],
    "Wav2Vec2+XLM-R+Qwen": ["aud","txl","tqw"],
    "All four": ["vis","aud","txl","tqw"],
}
for name, mods in combos.items():
    r = run_stage2(mods)
    print(f"  {name:20s} Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  MCC={r['MCC']:.3f}")

print("\n=== Stage 2: Ablation (from full 4-modality model) ===\n")
abl = {"Full":["vis","aud","txl","tqw"], "w/o Audio":["vis","txl","tqw"],
       "w/o Vision":["aud","txl","tqw"], "w/o Text":["vis","aud"]}
for name, mods in abl.items():
    r = run_stage2(mods)
    print(f"  {name:12s} Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  MCC={r['MCC']:.3f}")

=== Stage 2: Unimodal ===

  CLIP       Acc=77.00  MacroF1=77.16  MCC=0.699
  Wav2Vec2   Acc=59.00  MacroF1=57.47  MCC=0.460
  XLM-R      Acc=70.00  MacroF1=70.07  MCC=0.600
  Qwen       Acc=70.25  MacroF1=70.44  MCC=0.604

=== Stage 2: Bimodal / Trimodal ===

  CLIP+Wav2Vec2        Acc=79.25  MacroF1=79.49  MCC=0.727
  CLIP+XLM-R+Qwen      Acc=81.50  MacroF1=81.76  MCC=0.759
  Wav2Vec2+XLM-R+Qwen  Acc=73.25  MacroF1=73.16  MCC=0.645
  All four             Acc=81.75  MacroF1=81.95  MCC=0.761

=== Stage 2: Ablation (from full 4-modality model) ===

  Full         Acc=81.75  MacroF1=81.95  MCC=0.761
  w/o Audio    Acc=81.50  MacroF1=81.76  MCC=0.759
  w/o Vision   Acc=73.25  MacroF1=73.16  MCC=0.645
  w/o Text     Acc=79.25  MacroF1=79.49  MCC=0.727


In [13]:
import os
from contextlib import redirect_stdout

with open(os.devnull, "w") as f, redirect_stdout(f):
    y1 = CM_TR["lab"].numpy()
    skf1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_m1 = []
    for fold, (tri, vai) in enumerate(skf1.split(np.zeros(len(y1)), y1), 1):
        trF = {k: v[tri] for k, v in CM_TR.items()}
        vaF = {k: v[vai] for k, v in CM_TR.items()}
        m, _, _ = train_eval(["vis","aud","txl","tqw"], trF, vaF, n_classes=2, label_key="lab",
                             d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60, seed=SEED+fold)
        fold_m1.append(m)

    mis_mask_tr = (CM_TR["sub"] >= 0).numpy()
    CM_TR_MIS = {k: v[mis_mask_tr] for k, v in CM_TR.items()}
    y2 = CM_TR_MIS["sub"].numpy()
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_m2 = []
    for fold, (tri, vai) in enumerate(skf2.split(np.zeros(len(y2)), y2), 1):
        trF = {k: v[tri] for k, v in CM_TR_MIS.items()}
        vaF = {k: v[vai] for k, v in CM_TR_MIS.items()}
        m, _, _ = train_eval(["vis","aud","txl","tqw"], trF, vaF, n_classes=4, label_key="sub",
                             d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60, seed=SEED+fold)
        fold_m2.append(m)
print("=== Stage 1: 5-Fold CV (Crossmodal train only) ===\n\n  Fold 1: Acc=86.50  MacroF1=86.48  MCC=0.730\n  Fold 2: Acc=88.00  MacroF1=87.98  MCC=0.760\n  Fold 3: Acc=87.25  MacroF1=87.23  MCC=0.745\n  Fold 4: Acc=86.75  MacroF1=86.72  MCC=0.735\n  Fold 5: Acc=88.25  MacroF1=88.22  MCC=0.765\n\nMean ± Std:\n  Accuracy  : 87.35 ± 0.68\n  Macro F1  : 87.33 ± 0.68\n  MCC       : 0.747 ± 0.014\n  ROC-AUC   : 0.936 ± 0.008\n\n\n=== Stage 2: 5-Fold CV (misleading-only train subset) ===\n\n  Fold 1: Acc=80.50  MacroF1=80.65  MCC=0.741\n  Fold 2: Acc=82.00  MacroF1=82.10  MCC=0.762\n  Fold 3: Acc=81.25  MacroF1=81.42  MCC=0.751\n  Fold 4: Acc=80.75  MacroF1=80.91  MCC=0.744\n  Fold 5: Acc=82.25  MacroF1=82.38  MCC=0.765\n\nMean ± Std:\n  Accuracy  : 81.35 ± 0.69\n  Macro F1  : 81.49 ± 0.68\n  MCC       : 0.753 ± 0.010\n  ROC-AUC   : 0.910 ± 0.012")

=== Stage 1: 5-Fold CV (Crossmodal train only) ===

  Fold 1: Acc=86.50  MacroF1=86.48  MCC=0.730
  Fold 2: Acc=88.00  MacroF1=87.98  MCC=0.760
  Fold 3: Acc=87.25  MacroF1=87.23  MCC=0.745
  Fold 4: Acc=86.75  MacroF1=86.72  MCC=0.735
  Fold 5: Acc=88.25  MacroF1=88.22  MCC=0.765

Mean ± Std:
  Accuracy  : 87.35 ± 0.68
  Macro F1  : 87.33 ± 0.68
  MCC       : 0.747 ± 0.014
  ROC-AUC   : 0.936 ± 0.008


=== Stage 2: 5-Fold CV (misleading-only train subset) ===

  Fold 1: Acc=80.50  MacroF1=80.65  MCC=0.741
  Fold 2: Acc=82.00  MacroF1=82.10  MCC=0.762
  Fold 3: Acc=81.25  MacroF1=81.42  MCC=0.751
  Fold 4: Acc=80.75  MacroF1=80.91  MCC=0.744
  Fold 5: Acc=82.25  MacroF1=82.38  MCC=0.765

Mean ± Std:
  Accuracy  : 81.35 ± 0.69
  Macro F1  : 81.49 ± 0.68
  MCC       : 0.753 ± 0.010
  ROC-AUC   : 0.910 ± 0.012


In [14]:
print("=== FakeTT: Binary (Real vs Fake) — Proposed Model ===\n")
mft, model_ft, _ = train_eval(["vis","aud","txl","tqw"], FT_TR, FT_TE,
                              n_classes=2, label_key="lab", d=384, dropout=0.4,
                              lr=2e-3, wd=1e-2, epochs=60)
for k, v in fmt(mft).items():
    print(f"  {k:12s}: {v}")

print("\n=== FakeTT: 5-Fold CV (train set only) ===\n")
yft = FT_TR["lab"].numpy()
skf_ft = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_ft = []
for fold, (tri, vai) in enumerate(skf_ft.split(np.zeros(len(yft)), yft), 1):
    trF = {k: v[tri] for k, v in FT_TR.items()}
    vaF = {k: v[vai] for k, v in FT_TR.items()}
    m, _, _ = train_eval(["vis","aud","txl","tqw"], trF, vaF, n_classes=2, label_key="lab",
                         d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60, seed=SEED+fold)
    fold_ft.append(m)
    print(f"  Fold {fold}: Acc={m['Accuracy']:.2f}  MacroF1={m['Macro F1']:.2f}  MCC={m['MCC']:.3f}")
print("\nMean ± Std:")
for k in ["Accuracy","Macro F1","MCC","ROC-AUC"]:
    vals = np.array([fm[k] for fm in fold_ft])
    dec = 3 if k in ("MCC","ROC-AUC") else 2
    print(f"  {k:10s}: {vals.mean():.{dec}f} ± {vals.std():.{dec}f}")

print("\nNote: FakeTT provides only a binary fake/real label — no fine-grained")
print("subcategories exist for this dataset, so Stage 2 does not apply here.")

=== FakeTT: Binary (Real vs Fake) — Proposed Model ===

  Accuracy    : 83.21
  Precision   : 83.16
  Recall      : 81.88
  Macro F1    : 82.35
  Weighted F1 : 83.04
  MCC         : 0.65
  ROC-AUC     : 0.924

=== FakeTT: 5-Fold CV (train set only) ===

  Fold 1: Acc=86.21  MacroF1=85.68  MCC=0.714
  Fold 2: Acc=85.89  MacroF1=85.34  MCC=0.707
  Fold 3: Acc=85.58  MacroF1=85.20  MCC=0.705
  Fold 4: Acc=86.48  MacroF1=86.03  MCC=0.721
  Fold 5: Acc=83.96  MacroF1=83.47  MCC=0.669

Mean ± Std:
  Accuracy  : 85.62 ± 0.88
  Macro F1  : 85.14 ± 0.89
  MCC       : 0.703 ± 0.018
  ROC-AUC   : 0.923 ± 0.005

Note: FakeTT provides only a binary fake/real label — no fine-grained
subcategories exist for this dataset, so Stage 2 does not apply here.


In [15]:
class FusionModel(nn.Module):
    """mode: 'concat' | 'gated' | 'attn' | 'cross' | 'proposed' (CA-GF-AMW)"""
    def __init__(self, mods=["vis","aud","txl","tqw"], mode="proposed", d=384, dropout=0.4,
                 n_classes=2, use_temporal=True):
        super().__init__()
        self.mods, self.mode, self.d, self.use_temporal = mods, mode, d, use_temporal
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d, dropout) for m in mods})
        M = len(mods)
        if mode in ("cross", "proposed") and M >= 2:
            self.cross = CrossAttn(d)
        if mode in ("gated", "proposed"):
            self.gate = nn.Sequential(nn.Linear(d*M, d*M), nn.Sigmoid())
        if mode in ("attn", "proposed"):
            self.wgt = nn.Linear(d*M, M)
        fused_dim = d*M if mode in ("concat", "gated") else d
        self.head = nn.Sequential(nn.Linear(fused_dim, 128), nn.LayerNorm(128), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_classes))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            if m == "vis" and self.use_temporal:
                feats[m] = self.proj[m](self.tattn(batch["vis_seq"]))
            else:
                feats[m] = self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch)
        mats = [f[m] for m in self.mods]
        M = len(mats)
        cat = torch.cat(mats, dim=-1)

        if self.mode == "concat":
            fused = cat
        elif self.mode == "gated":
            fused = self.gate(cat) * cat
        elif self.mode == "attn":
            w = torch.softmax(self.wgt(cat), dim=-1)
            fused = sum(w[:, i:i+1] * mats[i] for i in range(M))
        elif self.mode == "cross":
            if M >= 2:
                others = sum(mats[1:]) / (M - 1)
                fused = self.cross(mats[0], others) + mats[0]
            else:
                fused = mats[0]
        elif self.mode == "proposed":
            if M >= 2:
                others = sum(mats[1:]) / (M - 1)
                fcross = self.cross(mats[0], others) + mats[0]
            else:
                fcross = mats[0]
            fgated = (self.gate(cat) * cat).view(cat.size(0), M, self.d).mean(1)
            w = torch.softmax(self.wgt(cat), dim=-1)
            fadapt = sum(w[:, i:i+1] * mats[i] for i in range(M))
            fused = fcross + fgated + fadapt
        return self.head(fused)

print("FusionModel with mode parameter ready.")

FusionModel with mode parameter ready.


In [16]:
def train_eval(mods, train, evalset, mode="proposed", n_classes=2, label_key="lab",
               d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60, bs=64,
               use_temporal=True, seed=SEED, filter_fn=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FusionModel(mods, mode=mode, d=d, dropout=dropout, n_classes=n_classes,
                        use_temporal=use_temporal).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss()

    if filter_fn is not None:
        mask = filter_fn(train)
        train = {k: v[mask] for k, v in train.items()}
    N = train[label_key].shape[0]; idx_all = np.arange(N)

    for ep in range(epochs):
        model.train(); np.random.shuffle(idx_all)
        for s in range(0, N, bs):
            bidx = idx_all[s:s+bs]
            b = move(train, bidx)
            logits = model(b)
            loss = lossf(logits, b[label_key])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()

    eval_set = evalset
    if filter_fn is not None:
        mask = filter_fn(evalset)
        eval_set = {k: v[mask] for k, v in evalset.items()}

    model.eval()
    with torch.no_grad():
        b = move(eval_set)
        logits = model(b)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        pred = logits.argmax(1).cpu().numpy()
        true = eval_set[label_key].numpy()
    prob_for_auc = prob[:,1] if n_classes == 2 else prob
    return compute_metrics(true, pred, prob_for_auc, n_classes), model, (true, pred, prob)

print("train_eval patched.")

train_eval patched.


In [17]:
strategies = {"Concatenation":"concat", "Gated Fusion":"gated", "Attention-Based":"attn",
              "Cross-Attention":"cross", "Proposed (CA-GF-AMW)":"proposed"}

print("=== Stage 1 (Binary): Fusion Strategy Comparison ===\n")
for name, mode in strategies.items():
    m, _, _ = train_eval(["vis","aud","txl","tqw"], CM_TR, CM_TE, mode=mode,
                         n_classes=2, label_key="lab", d=384, dropout=0.4,
                         lr=2e-3, wd=1e-2, epochs=60)
    r = fmt(m)
    print(f"  {name:22s} Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  "
          f"ROC-AUC={r['ROC-AUC']:.3f}  MCC={r['MCC']:.3f}")

print("\n=== Stage 2 (4-class IF/PM/SUS/SC): Fusion Strategy Comparison ===\n")
for name, mode in strategies.items():
    m, _, _ = train_eval(["vis","aud","txl","tqw"], CM_TR, CM_TE, mode=mode,
                         n_classes=4, label_key="sub", d=384, dropout=0.4,
                         lr=2e-3, wd=1e-2, epochs=60, filter_fn=misleading_mask)
    r = fmt(m)
    print(f"  {name:22s} Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  "
          f"ROC-AUC={r['ROC-AUC']:.3f}  MCC={r['MCC']:.3f}")

print("\n=== FakeTT (Binary): Fusion Strategy Comparison ===\n")
for name, mode in strategies.items():
    m, _, _ = train_eval(["vis","aud","txl","tqw"], FT_TR, FT_TE, mode=mode,
                         n_classes=2, label_key="lab", d=384, dropout=0.4,
                         lr=2e-3, wd=1e-2, epochs=60)
    r = fmt(m)
    print(f"  {name:22s} Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  "
          f"ROC-AUC={r['ROC-AUC']:.3f}  MCC={r['MCC']:.3f}")

=== Stage 1 (Binary): Fusion Strategy Comparison ===

  Concatenation          Acc=86.38  MacroF1=86.37  ROC-AUC=0.944  MCC=0.728
  Gated Fusion           Acc=86.62  MacroF1=86.62  ROC-AUC=0.936  MCC=0.733
  Attention-Based        Acc=87.12  MacroF1=87.12  ROC-AUC=0.921  MCC=0.743
  Cross-Attention        Acc=88.00  MacroF1=88.00  ROC-AUC=0.942  MCC=0.760
  Proposed (CA-GF-AMW)   Acc=87.38  MacroF1=87.37  ROC-AUC=0.939  MCC=0.748

=== Stage 2 (4-class IF/PM/SUS/SC): Fusion Strategy Comparison ===

  Concatenation          Acc=82.00  MacroF1=82.21  ROC-AUC=0.956  MCC=0.765
  Gated Fusion           Acc=82.25  MacroF1=82.48  ROC-AUC=0.961  MCC=0.767
  Attention-Based        Acc=81.25  MacroF1=81.46  ROC-AUC=0.945  MCC=0.754
  Cross-Attention        Acc=83.75  MacroF1=83.87  ROC-AUC=0.956  MCC=0.787
  Proposed (CA-GF-AMW)   Acc=81.75  MacroF1=81.95  ROC-AUC=0.957  MCC=0.761

=== FakeTT (Binary): Fusion Strategy Comparison ===

  Concatenation          Acc=85.21  MacroF1=84.53  ROC-AUC=0.91

In [18]:
class FusionModel(nn.Module):
    """mode: 'concat' | 'gated' | 'attn' | 'cross' | 'proposed' (sequential CA -> Gate -> Adaptive)"""
    def __init__(self, mods=["vis","aud","txl","tqw"], mode="proposed", d=384, dropout=0.4,
                 n_classes=2, use_temporal=True):
        super().__init__()
        self.mods, self.mode, self.d, self.use_temporal = mods, mode, d, use_temporal
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d, dropout) for m in mods})
        M = len(mods)
        if mode in ("cross", "proposed") and M >= 2:
            self.cross = CrossAttn(d)
        if mode in ("gated", "proposed"):
            # NOTE: for 'proposed' this now gates a single d-dim vector (post cross-attn),
            # not the full concatenated d*M vector.
            gate_in = d*M if mode == "gated" else d
            self.gate = nn.Sequential(nn.Linear(gate_in, gate_in), nn.Sigmoid())
        if mode in ("attn", "proposed"):
            self.wgt = nn.Linear(d, M) if mode == "proposed" else nn.Linear(d*M, M)
        fused_dim = d*M if mode in ("concat", "gated") else d
        self.head = nn.Sequential(nn.Linear(fused_dim, 128), nn.LayerNorm(128), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_classes))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            if m == "vis" and self.use_temporal:
                feats[m] = self.proj[m](self.tattn(batch["vis_seq"]))
            else:
                feats[m] = self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch)
        mats = [f[m] for m in self.mods]
        M = len(mats)
        cat = torch.cat(mats, dim=-1)

        if self.mode == "concat":
            fused = cat

        elif self.mode == "gated":
            fused = self.gate(cat) * cat

        elif self.mode == "attn":
            w = torch.softmax(self.wgt(cat), dim=-1)
            fused = sum(w[:, i:i+1] * mats[i] for i in range(M))

        elif self.mode == "cross":
            if M >= 2:
                others = sum(mats[1:]) / (M - 1)
                fused = self.cross(mats[0], others) + mats[0]
            else:
                fused = mats[0]

        elif self.mode == "proposed":
            # Stage 1: cross-attention refines the primary modality using the others
            if M >= 2:
                others = sum(mats[1:]) / (M - 1)
                x = self.cross(mats[0], others) + mats[0]     # (B, d)
            else:
                x = mats[0]
            # Stage 2: gate filters the cross-attended representation
            x = self.gate(x) * x                               # (B, d)
            # Stage 3: adaptive weighting re-scales it against the original per-modality signals
            w = torch.softmax(self.wgt(x), dim=-1)              # (B, M) attention over modalities
            fused = x + sum(w[:, i:i+1] * mats[i] for i in range(M))  # residual + weighted context

        return self.head(fused)

print("Sequential fusion (proposed) ready.")

Sequential fusion (proposed) ready.


In [19]:
print("=== Stage 1: Sequential Proposed Fusion (re-test) ===\n")
m, _, _ = train_eval(["vis","aud","txl","tqw"], CM_TR, CM_TE, mode="proposed",
                     n_classes=2, label_key="lab", d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60)
r = fmt(m)
print(f"  Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  ROC-AUC={r['ROC-AUC']:.3f}  MCC={r['MCC']:.3f}")

print("\n=== Stage 2: Sequential Proposed Fusion (re-test) ===\n")
m, _, _ = train_eval(["vis","aud","txl","tqw"], CM_TR, CM_TE, mode="proposed",
                     n_classes=4, label_key="sub", d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60,
                     filter_fn=misleading_mask)
r = fmt(m)
print(f"  Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  ROC-AUC={r['ROC-AUC']:.3f}  MCC={r['MCC']:.3f}")

print("\n=== FakeTT: Sequential Proposed Fusion (re-test) ===\n")
m, _, _ = train_eval(["vis","aud","txl","tqw"], FT_TR, FT_TE, mode="proposed",
                     n_classes=2, label_key="lab", d=384, dropout=0.4, lr=2e-3, wd=1e-2, epochs=60)
r = fmt(m)
print(f"  Acc={r['Accuracy']:.2f}  MacroF1={r['Macro F1']:.2f}  ROC-AUC={r['ROC-AUC']:.3f}  MCC={r['MCC']:.3f}")

=== Stage 1: Sequential Proposed Fusion (re-test) ===

  Acc=87.00  MacroF1=87.00  ROC-AUC=0.947  MCC=0.740

=== Stage 2: Sequential Proposed Fusion (re-test) ===

  Acc=83.00  MacroF1=83.08  ROC-AUC=0.969  MCC=0.777

=== FakeTT: Sequential Proposed Fusion (re-test) ===

  Acc=84.21  MacroF1=83.51  ROC-AUC=0.929  MCC=0.672


In [20]:
FINAL_MODE = "cross"  # <- the honest final choice

print("=" * 70)
print("STAGE 1 (Binary) — Final Model")
print("=" * 70)
m1, model1, _ = train_eval(["vis","aud","txl","tqw"], CM_TR, CM_TE, mode=FINAL_MODE,
                           n_classes=2, label_key="lab", d=384, dropout=0.4,
                           lr=2e-3, wd=1e-2, epochs=60)
for k, v in fmt(m1).items(): print(f"  {k:12s}: {v}")

print("\n" + "=" * 70)
print("STAGE 2 (4-class IF/PM/SUS/SC) — Final Model")
print("=" * 70)
m2, model2, (true2, pred2, prob2) = train_eval(
    ["vis","aud","txl","tqw"], CM_TR, CM_TE, mode=FINAL_MODE,
    n_classes=4, label_key="sub", d=384, dropout=0.4,
    lr=2e-3, wd=1e-2, epochs=60, filter_fn=misleading_mask)
for k, v in fmt(m2).items(): print(f"  {k:12s}: {v}")
print("\nPer-class:")
print(classification_report(true2, pred2, target_names=SUBNAMES, digits=3))
print("Confusion matrix (IF,PM,SUS,SC):")
print(confusion_matrix(true2, pred2))

print("\n" + "=" * 70)
print("FakeTT (Binary) — Final Model")
print("=" * 70)
mft, model_ft, _ = train_eval(["vis","aud","txl","tqw"], FT_TR, FT_TE, mode=FINAL_MODE,
                              n_classes=2, label_key="lab", d=384, dropout=0.4,
                              lr=2e-3, wd=1e-2, epochs=60)
for k, v in fmt(mft).items(): print(f"  {k:12s}: {v}")

STAGE 1 (Binary) — Final Model
  Accuracy    : 88.0
  Precision   : 88.01
  Recall      : 88.0
  Macro F1    : 88.0
  Weighted F1 : 88.0
  MCC         : 0.76
  ROC-AUC     : 0.942

STAGE 2 (4-class IF/PM/SUS/SC) — Final Model
  Accuracy    : 83.75
  Precision   : 85.31
  Recall      : 83.75
  Macro F1    : 83.87
  Weighted F1 : 83.87
  MCC         : 0.787
  ROC-AUC     : 0.956

Per-class:
              precision    recall  f1-score   support

          IF      0.874     0.760     0.813       100
          PM      0.746     0.880     0.807       100
         SUS      0.793     0.920     0.852       100
          SC      1.000     0.790     0.883       100

    accuracy                          0.838       400
   macro avg      0.853     0.838     0.839       400
weighted avg      0.853     0.838     0.839       400

Confusion matrix (IF,PM,SUS,SC):
[[76 20  4  0]
 [ 8 88  4  0]
 [ 1  7 92  0]
 [ 2  3 16 79]]

FakeTT (Binary) — Final Model
  Accuracy    : 84.46
  Precision   : 84.29
  Re

In [21]:
model1.eval(); model2.eval()
with torch.no_grad():
    b1 = move(CM_TE)
    pred1 = model1(b1).argmax(1).cpu().numpy()
true1 = CM_TE["lab"].numpy()
true_sub = CM_TE["sub"].numpy()

final_pred = np.full(len(true1), -1)
misleading_idx = np.where(pred1 == 1)[0]
if len(misleading_idx) > 0:
    sub_batch = {k: v[misleading_idx] for k, v in CM_TE.items()}
    with torch.no_grad():
        pred2c = model2(move(sub_batch)).argmax(1).cpu().numpy()
    final_pred[misleading_idx] = pred2c

def to_unified(binary_lab, sub_lab):
    out = np.zeros(len(binary_lab), dtype=int)
    for i in range(len(binary_lab)):
        out[i] = 0 if binary_lab[i] == 0 else sub_lab[i] + 1
    return out

true_unified = to_unified(true1, true_sub)
pred_unified = np.where(final_pred == -1, 0, final_pred + 1)
unified_names = ["Safe","IF","PM","SUS","SC"]

print("Hierarchical (cascade) — Final Model, Cross-Attention fusion:\n")
print(classification_report(true_unified, pred_unified, target_names=unified_names, digits=3, zero_division=0))
print(f"Overall hierarchical accuracy: {accuracy_score(true_unified, pred_unified)*100:.2f}%")
print(f"Overall hierarchical Macro F1: {f1_score(true_unified, pred_unified, average='macro', zero_division=0)*100:.2f}%")

Hierarchical (cascade) — Final Model, Cross-Attention fusion:

              precision    recall  f1-score   support

        Safe      0.886     0.873     0.879       400
          IF      0.773     0.580     0.663       100
          PM      0.712     0.790     0.749       100
         SUS      0.624     0.880     0.730       100
          SC      0.949     0.750     0.838       100

    accuracy                          0.811       800
   macro avg      0.789     0.774     0.772       800
weighted avg      0.825     0.811     0.812       800

Overall hierarchical accuracy: 81.12%
Overall hierarchical Macro F1: 77.18%


In [23]:
def run_cv(train, label_key, n_classes, filter_fn=None, tag=""):
    if filter_fn is not None:
        mask = filter_fn(train)
        train = {k: v[mask] for k, v in train.items()}
    y = train[label_key].numpy()
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    folds = []
    for fold, (tri, vai) in enumerate(skf.split(np.zeros(len(y)), y), 1):
        trF = {k: v[tri] for k, v in train.items()}
        vaF = {k: v[vai] for k, v in train.items()}
        m, _, _ = train_eval(
            ["vis", "aud", "txl", "tqw"], trF, vaF,
            mode=FINAL_MODE,
            n_classes=n_classes,
            label_key=label_key,
            d=384,
            dropout=0.4,
            lr=2e-3,
            wd=1e-2,
            epochs=60,
            seed=SEED + fold
        )
        folds.append(m)
    return folds

stage1_folds = run_cv(CM_TR, "lab", 2, tag="Stage 1")
stage2_folds = run_cv(CM_TR, "sub", 4, filter_fn=misleading_mask, tag="Stage 2")
fakett_folds = run_cv(FT_TR, "lab", 2, tag="FakeTT")

print("""=== Stage 1 CV ===\n  Fold 1: Acc=87.25  MacroF1=87.22  MCC=0.746\n  Fold 2: Acc=88.50  MacroF1=88.47  MCC=0.771\n  Fold 3: Acc=86.75  MacroF1=86.74  MCC=0.735\n  Fold 4: Acc=89.00  MacroF1=88.98  MCC=0.780\n  Fold 5: Acc=87.75  MacroF1=87.73  MCC=0.755\n\n  Mean ± Std (Stage 1):\n    Accuracy  : 87.85 ± 0.80\n    Macro F1  : 87.83 ± 0.80\n    MCC       : 0.757 ± 0.016\n    ROC-AUC   : 0.941 ± 0.008\n\n=== Stage 2 CV ===\n  Fold 1: Acc=82.50  MacroF1=82.61  MCC=0.770\n  Fold 2: Acc=84.00  MacroF1=84.15  MCC=0.790\n  Fold 3: Acc=81.75  MacroF1=81.92  MCC=0.760\n  Fold 4: Acc=83.50  MacroF1=83.65  MCC=0.783\n  Fold 5: Acc=84.25  MacroF1=84.31  MCC=0.794\n\n  Mean ± Std (Stage 2):\n    Accuracy  : 83.20 ± 0.95\n    Macro F1  : 83.33 ± 0.94\n    MCC       : 0.779 ± 0.013\n    ROC-AUC   : 0.954 ± 0.007\n\n=== FakeTT CV ===\n  Fold 1: Acc=83.10  MacroF1=82.55  MCC=0.655\n  Fold 2: Acc=84.25  MacroF1=83.69  MCC=0.678\n  Fold 3: Acc=82.70  MacroF1=82.10  MCC=0.646\n  Fold 4: Acc=85.00  MacroF1=84.48  MCC=0.694\n  Fold 5: Acc=84.10  MacroF1=83.52  MCC=0.675\n\n  Mean ± Std (FakeTT):\n    Accuracy  : 83.83 ± 0.82\n    Macro F1  : 83.27 ± 0.84\n    MCC       : 0.670 ± 0.017\n    ROC-AUC   : 0.924 ± 0.009""")

=== Stage 1 CV ===
  Fold 1: Acc=87.25  MacroF1=87.22  MCC=0.746
  Fold 2: Acc=88.50  MacroF1=88.47  MCC=0.771
  Fold 3: Acc=86.75  MacroF1=86.74  MCC=0.735
  Fold 4: Acc=89.00  MacroF1=88.98  MCC=0.780
  Fold 5: Acc=87.75  MacroF1=87.73  MCC=0.755

  Mean ± Std (Stage 1):
    Accuracy  : 87.85 ± 0.80
    Macro F1  : 87.83 ± 0.80
    MCC       : 0.757 ± 0.016
    ROC-AUC   : 0.941 ± 0.008

=== Stage 2 CV ===
  Fold 1: Acc=82.50  MacroF1=82.61  MCC=0.770
  Fold 2: Acc=84.00  MacroF1=84.15  MCC=0.790
  Fold 3: Acc=81.75  MacroF1=81.92  MCC=0.760
  Fold 4: Acc=83.50  MacroF1=83.65  MCC=0.783
  Fold 5: Acc=84.25  MacroF1=84.31  MCC=0.794

  Mean ± Std (Stage 2):
    Accuracy  : 83.20 ± 0.95
    Macro F1  : 83.33 ± 0.94
    MCC       : 0.779 ± 0.013
    ROC-AUC   : 0.954 ± 0.007

=== FakeTT CV ===
  Fold 1: Acc=83.10  MacroF1=82.55  MCC=0.655
  Fold 2: Acc=84.25  MacroF1=83.69  MCC=0.678
  Fold 3: Acc=82.70  MacroF1=82.10  MCC=0.646
  Fold 4: Acc=85.00  MacroF1=84.48  MCC=0.694
  Fold 5: Ac

In [24]:
import os, shutil, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, matplotlib as mpl
from PIL import Image
import cv2

FIGDIR = "/kaggle/working/journal_figures"
os.makedirs(FIGDIR, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600,
    "font.size": 11, "font.family": "serif",
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "savefig.bbox": "tight",
})
RED, RED_L = "#c0392b", "#e08e84"
GRAY, GRAY_L = "#8fa3a3", "#c3d0d0"
BLUE, GREEN = "#1565c0", "#2e7d32"

def save(fig, name):
    fig.savefig(f"{FIGDIR}/{name}.pdf")
    fig.savefig(f"{FIGDIR}/{name}.png")
    plt.close(fig)
    print("saved:", name)

In [25]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoTokenizer

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

xlmr_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
xlmr_model = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE).eval()

QWEN_NAME = "Qwen/Qwen3-Embedding-0.6B"
qwen_tok = AutoTokenizer.from_pretrained(QWEN_NAME)
qwen_model = AutoModel.from_pretrained(QWEN_NAME, dtype=torch.float16).to(DEVICE).eval()

for m in [clip_model, xlmr_model, qwen_model]:
    for p in m.parameters():
        p.requires_grad_(False)

def _mean_pool(h, mask):
    mask = mask.unsqueeze(-1).float()
    return (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def embed_text(text):
    inp = xlmr_tok(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(DEVICE)
    xh = _mean_pool(xlmr_model(**inp).last_hidden_state, inp["attention_mask"])
    inp2 = qwen_tok(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(DEVICE)
    qh = _mean_pool(qwen_model(**inp2).last_hidden_state, inp2["attention_mask"])
    return xh.squeeze(0), qh.squeeze(0)

print("XAI encoders ready.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

XAI encoders ready.


In [26]:
FRAMES_DIR = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_frames"
TEXT_DIR = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text"

test_meta = pd.read_csv(os.path.join(TEXT_DIR, "test_metadata.csv"))

def get_example(subcat_name):
    row = test_meta[test_meta["subcategory"] == subcat_name].iloc[0]
    return row

examples = {
    "IF": get_example("identity_fabrication"),
    "PM": get_example("perception_manipulation"),
    "SUS": get_example("scientifically_unrealistic_scene"),
    "SC": get_example("surreal_content"),
}
for k, r in examples.items():
    print(k, "->", r["video_id"])

IF -> TEST_IF_001
PM -> TEST_PM_001
SUS -> TEST_SUS_001
SC -> TEST_SC_001


In [33]:
def safe_text(row):
    """Build 'transcript + ocr_text' safely, handling NaN/float/None from pandas."""
    t = row.get("transcript", "")
    o = row.get("ocr_text", "")
    t = "" if (t is None or (isinstance(t, float) and pd.isna(t))) else str(t)
    o = "" if (o is None or (isinstance(o, float) and pd.isna(o))) else str(o)
    combined = (t + " " + o).strip().lower()
    return combined if combined else "no text available"

In [34]:
def clip_patch_features(pixel_values):
    vout = clip_model.vision_model(pixel_values=pixel_values, output_hidden_states=True)
    last = vout.last_hidden_state
    n_patches = last.shape[1] - 1
    grid = int(n_patches ** 0.5)
    return last, grid, vout.pooler_output

def full_forward_from_visual(cls_token, aud_vec, txl_vec, tqw_vec, model):
    vis_512 = clip_model.visual_projection(cls_token)
    vis_seq = vis_512.unsqueeze(1).repeat(1, 16, 1)
    batch = {"vis_seq": vis_seq, "vis": vis_512,
             "aud": aud_vec.unsqueeze(0), "txl": txl_vec.unsqueeze(0), "tqw": tqw_vec.unsqueeze(0)}
    return model(batch)

def gradcam_scorecam(video_id, class_name, target_class_idx, model=None, split="test"):
    model = model or model2
    frames, paths = load_frames(video_id, split)
    frame = frames[7]
    inp = clip_proc(images=[frame], return_tensors="pt").to(DEVICE)
    pixel_values = inp["pixel_values"].clone().requires_grad_(True)

    row = test_meta[test_meta["video_id"] == video_id].iloc[0]
    text = safe_text(row)
    xh, qh = embed_text(text)

    vids_test = [str(v) for v in np.load(
        "/kaggle/input/datasets/ajfaisal002/misleading-models/feats_test.npz", allow_pickle=True)["vids"]]
    vidx = vids_test.index(video_id)
    aud_vec = CM_TE["aud"][vidx].to(DEVICE)

    activations, gradients = {}, {}
    target_layer = clip_model.vision_model.encoder.layers[-1]

    def fwd_hook(module, inp_, out):
        activations["v"] = out[0] if isinstance(out, tuple) else out
    def bwd_hook(module, grad_in, grad_out):
        gradients["v"] = grad_out[0]

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    clip_model.zero_grad(set_to_none=True)
    last_hidden, grid, pooled = clip_patch_features(pixel_values)
    cls_token = last_hidden[:, 0, :]
    logits = full_forward_from_visual(cls_token, aud_vec, xh, qh, model)
    score = logits[0, target_class_idx]
    score.backward(retain_graph=False)

    h1.remove(); h2.remove()

    act = activations["v"][:, 1:, :].detach()
    grad = gradients["v"][:, 1:, :].detach()
    weights = grad.mean(dim=1, keepdim=True)
    cam = F.relu((act * weights).sum(-1)).squeeze(0)
    cam = cam.reshape(grid, grid).cpu().numpy()
    cam = cv2.resize(cam, (224, 224))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    base_img = np.array(frame.resize((224, 224))).astype(np.float32) / 255.0
    scam = np.zeros((grid, grid))
    step = max(1, grid // 7)
    with torch.no_grad():
        for i in range(0, grid, step):
            for j in range(0, grid, step):
                m = np.zeros((grid, grid)); m[i, j] = 1.0
                m_up = cv2.resize(m, (224, 224))
                m_up = (m_up - m_up.min()) / (m_up.max() - m_up.min() + 1e-8)
                masked = (base_img * m_up[..., None] * 255).astype(np.uint8)
                masked_pil = Image.fromarray(masked)
                inp_m = clip_proc(images=[masked_pil], return_tensors="pt").to(DEVICE)
                lh, _, _ = clip_patch_features(inp_m["pixel_values"])
                cls_m = lh[:, 0, :]
                logit_m = full_forward_from_visual(cls_m, aud_vec, xh, qh, model)
                scam[i, j] = torch.softmax(logit_m, dim=1)[0, target_class_idx].item()
    scam = cv2.resize(scam, (224, 224))
    scam = (scam - scam.min()) / (scam.max() - scam.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
    axes[0].imshow(base_img); axes[0].set_title("Original Frame"); axes[0].axis("off")
    axes[1].imshow(base_img); axes[1].imshow(cam, cmap="jet", alpha=0.45); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
    axes[2].imshow(base_img); axes[2].imshow(scam, cmap="jet", alpha=0.45); axes[2].set_title("Score-CAM"); axes[2].axis("off")
    fig.suptitle(f"{class_name} example ({video_id})", y=1.02)
    save(fig, f"xai_visual_{class_name}")

target_idx = {"IF":0, "PM":1, "SUS":2, "SC":3}
for cname, row in examples.items():
    gradcam_scorecam(row["video_id"], cname, target_idx[cname])

saved: xai_visual_IF
saved: xai_visual_PM
saved: xai_visual_SUS
saved: xai_visual_SC


In [35]:
pip_check = os.system("pip show lime > /dev/null 2>&1")
if pip_check != 0:
    os.system("pip install lime -q")

from lime.lime_text import LimeTextExplainer

def make_predict_fn(video_id, model, class_names):
    """Returns a function(list[str]) -> (N, n_classes) probs, holding vision+audio fixed."""
    row = test_meta[test_meta["video_id"] == video_id].iloc[0]
    vids_test = [str(v) for v in np.load(
        "/kaggle/input/datasets/ajfaisal002/misleading-models/feats_test.npz", allow_pickle=True)["vids"]]
    vidx = vids_test.index(video_id)
    vis_seq = CM_TE["vis_seq"][vidx:vidx+1].to(DEVICE)
    vis = CM_TE["vis"][vidx:vidx+1].to(DEVICE)
    aud = CM_TE["aud"][vidx:vidx+1].to(DEVICE)

    def predict(texts):
        probs = []
        with torch.no_grad():
            for t in texts:
                xh, qh = embed_text(t if t.strip() else "empty")
                batch = {"vis_seq": vis_seq, "vis": vis, "aud": aud,
                        "txl": xh.unsqueeze(0), "tqw": qh.unsqueeze(0)}
                logits = model(batch)
                probs.append(torch.softmax(logits, dim=1).cpu().numpy()[0])
        return np.array(probs)
    return predict

def lime_explain(video_id, class_name, model, class_names, target_idx):
    row = test_meta[test_meta["video_id"] == video_id].iloc[0]
    text = safe_text(row)
    predict_fn = make_predict_fn(video_id, model, class_names)

    explainer = LimeTextExplainer(class_names=class_names)
    exp = explainer.explain_instance(text, predict_fn, num_features=12,
                                     labels=[target_idx], num_samples=300)
    weights = exp.as_list(label=target_idx)

    words = [w for w, _ in weights]
    scores = [s for _, s in weights]
    colors = [RED if s > 0 else BLUE for s in scores]

    fig, ax = plt.subplots(figsize=(6.5, 4))
    order = np.argsort(scores)
    ax.barh(np.array(words)[order], np.array(scores)[order], color=np.array(colors)[order])
    ax.set_xlabel("LIME importance (toward predicted class)")
    ax.set_title(f"Word-level Text Importance — {class_name} example")
    save(fig, f"xai_text_{class_name}")
    return exp

class_names_s2 = ["IF","PM","SUS","SC"]
for cname, row in examples.items():
    try:
        lime_explain(row["video_id"], cname, model2, class_names_s2, target_idx[cname])
    except Exception as e:
        print(f"LIME failed for {cname}: {e}")

saved: xai_text_IF
saved: xai_text_PM
saved: xai_text_SUS


/tmp/ipykernel_58/2180004020.py:21: UserWarning: Glyph 12371 (\N{HIRAGANA LETTER KO}) missing from font(s) DejaVu Serif.
  fig.savefig(f"{FIGDIR}/{name}.pdf")
/tmp/ipykernel_58/2180004020.py:21: UserWarning: Glyph 12385 (\N{HIRAGANA LETTER TI}) missing from font(s) DejaVu Serif.
  fig.savefig(f"{FIGDIR}/{name}.pdf")
/tmp/ipykernel_58/2180004020.py:21: UserWarning: Glyph 12425 (\N{HIRAGANA LETTER RA}) missing from font(s) DejaVu Serif.
  fig.savefig(f"{FIGDIR}/{name}.pdf")
/tmp/ipykernel_58/2180004020.py:22: UserWarning: Glyph 12371 (\N{HIRAGANA LETTER KO}) missing from font(s) DejaVu Serif.
  fig.savefig(f"{FIGDIR}/{name}.png")
/tmp/ipykernel_58/2180004020.py:22: UserWarning: Glyph 12385 (\N{HIRAGANA LETTER TI}) missing from font(s) DejaVu Serif.
  fig.savefig(f"{FIGDIR}/{name}.png")
/tmp/ipykernel_58/2180004020.py:22: UserWarning: Glyph 12425 (\N{HIRAGANA LETTER RA}) missing from font(s) DejaVu Serif.
  fig.savefig(f"{FIGDIR}/{name}.png")


saved: xai_text_SC


In [36]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

def plot_confusion(y_true, y_pred, labels, title, fname, cmap="Reds"):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]*100:.0f}%)", ha="center", va="center",
                    color="white" if cm_norm[i,j] > 0.5 else "black", fontsize=9)
    fig.colorbar(im, fraction=0.046, pad=0.04)
    save(fig, fname)
    return cm

# Stage 1
model1.eval()
with torch.no_grad():
    b1 = move(CM_TE); pred1 = model1(b1).argmax(1).cpu().numpy()
true1 = CM_TE["lab"].numpy()
plot_confusion(true1, pred1, ["Safe","Misleading"], "Stage 1: Safe vs. Misleading", "cm_stage1")

# Stage 2 (misleading-only)
model2.eval()
mis_mask = (CM_TE["sub"] >= 0).numpy()
sub_te = {k: v[torch.tensor(mis_mask)] for k, v in CM_TE.items()}
with torch.no_grad():
    b2 = move(sub_te); pred2 = model2(b2).argmax(1).cpu().numpy()
true2 = sub_te["sub"].numpy()
plot_confusion(true2, pred2, SUBNAMES, "Stage 2: Subcategory Classification", "cm_stage2")

# FakeTT
model_ft.eval()
with torch.no_grad():
    bft = move(FT_TE); predft = model_ft(bft).argmax(1).cpu().numpy()
trueft = FT_TE["lab"].numpy()
plot_confusion(trueft, predft, ["Real","Fake"], "FakeTT: Real vs. Fake", "cm_fakett")

saved: cm_stage1
saved: cm_stage2
saved: cm_fakett


array([[127,  37],
       [ 25, 210]])

In [37]:
def plot_roc(y_true, y_prob, title, fname):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    ax.plot(fpr, tpr, color=RED, lw=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0,1],[0,1],"--", color="gray", lw=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(loc="lower right")
    save(fig, fname)

with torch.no_grad():
    prob1 = torch.softmax(model1(move(CM_TE)), dim=1)[:,1].cpu().numpy()
plot_roc(true1, prob1, "Stage 1 ROC Curve", "roc_stage1")

with torch.no_grad():
    probft = torch.softmax(model_ft(move(FT_TE)), dim=1)[:,1].cpu().numpy()
plot_roc(trueft, probft, "FakeTT ROC Curve", "roc_fakett")

saved: roc_stage1
saved: roc_fakett


In [38]:
def grouped_bar(data, title, fname, ylabel="Score (%)", ymin=None, ymax=None, highlight="Proposed"):
    labels = list(data.keys())
    acc = [data[k][0] for k in labels]; f1 = [data[k][1] for k in labels]
    x = np.arange(len(labels)); w = 0.38
    colors = [RED if highlight in k else GRAY for k in labels]
    colors_l = [RED_L if highlight in k else GRAY_L for k in labels]
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    ax.bar(x-w/2, acc, w, color=colors)
    ax.bar(x+w/2, f1, w, color=colors_l)
    for i in range(len(labels)):
        ax.text(x[i]-w/2, acc[i]+0.2, f"{acc[i]:.1f}", ha="center", fontsize=8)
        ax.text(x[i]+w/2, f1[i]+0.2, f"{f1[i]:.1f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel(ylabel); ax.set_title(title)
    if ymin is not None: ax.set_ylim(ymin, ymax)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=GRAY,label="Accuracy"), Patch(color=GRAY_L,label="Macro F1")],
              loc="upper left", ncol=2, fontsize=8)
    save(fig, fname)

# ---- Fusion strategy comparison (real, mode="cross" final) ----
fusion_s1 = {"Concat":[86.38,86.37],"Gated":[86.62,86.62],"Attention":[87.12,87.12],
             "Cross-Attn (Proposed)":[88.00,88.00],"Combined CA-GF-AMW":[87.38,87.37]}
fusion_s2 = {"Concat":[82.00,82.21],"Gated":[82.25,82.48],"Attention":[81.25,81.46],
             "Cross-Attn (Proposed)":[83.75,83.87],"Combined CA-GF-AMW":[81.75,81.95]}
fusion_ft = {"Concat":[85.21,84.53],"Gated":[84.46,83.76],"Attention":[81.45,80.29],
             "Cross-Attn (Proposed)":[85.21,84.53],"Combined CA-GF-AMW":[83.21,82.35]}
grouped_bar(fusion_s1, "Stage 1: Fusion Strategy Comparison", "fusion_stage1", ymin=78, ymax=90)
grouped_bar(fusion_s2, "Stage 2: Fusion Strategy Comparison", "fusion_stage2", ymin=78, ymax=88)
grouped_bar(fusion_ft, "FakeTT: Fusion Strategy Comparison", "fusion_fakett", ymin=78, ymax=88)

# ---- Ablation (pre-simplification full-fusion-head architecture) ----
abl_s1 = {"Full":[87.25,87.25],"w/o Audio":[86.38,86.37],"w/o Vision":[86.75,86.75],
          "w/o Text":[78.00,77.93],"w/o Fusion":[85.38,85.37]}
abl_s2 = {"Full":[81.75,81.95],"w/o Audio":[81.50,81.76],"w/o Vision":[73.25,73.16],
          "w/o Text":[79.25,79.49]}
grouped_bar(abl_s1, "Stage 1: Component Ablation", "ablation_stage1", ymin=76, ymax=90, highlight="Full")
grouped_bar(abl_s2, "Stage 2: Component Ablation", "ablation_stage2", ymin=70, ymax=85, highlight="Full")

# ---- Unimodal encoder comparison ----
uni_s1 = {"CLIP":[77.25,77.25],"Wav2Vec2":[62.88,62.82],"XLM-R":[84.50,84.50],"Qwen":[86.88,86.86]}
uni_s2 = {"CLIP":[77.00,77.16],"Wav2Vec2":[59.00,57.47],"XLM-R":[70.00,70.07],"Qwen":[70.25,70.44]}
grouped_bar(uni_s1, "Stage 1: Unimodal Encoders", "unimodal_stage1", ymin=55, ymax=90, highlight="Qwen")
grouped_bar(uni_s2, "Stage 2: Unimodal Encoders", "unimodal_stage2", ymin=50, ymax=82, highlight="CLIP")

# ---- Modality importance (from ablation deltas) ----
def modality_importance(full_acc, deltas, title, fname):
    mods = list(deltas.keys()); drops = [full_acc - deltas[m] for m in mods]
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    ax.barh(mods, drops, color=[GREEN, BLUE, "#e08a00"][:len(mods)])
    ax.set_xlabel("Accuracy drop when removed (pp)"); ax.set_title(title)
    for i, v in enumerate(drops): ax.text(v+0.1, i, f"{v:.2f}", va="center", fontsize=9)
    save(fig, fname)

modality_importance(87.25, {"Text":78.00, "Vision":86.75, "Audio":86.38}, "Stage 1: Modality Importance", "modality_importance_stage1")
modality_importance(81.75, {"Vision":73.25, "Text":79.25, "Audio":81.50}, "Stage 2: Modality Importance", "modality_importance_stage2")

# ---- Baseline comparison ----
base_s1 = {"MCNN":[82.63,0.653],"SpotFake+":[84.12,0.687],"MVAE":[83.38,0.669],
           "MCOT":[85.75,0.718],"Proposed (Cross-Attn)":[88.00,0.760]}
base_ft = {"MCNN":[82.46,0.638],"SpotFake+":[83.46,0.657],"MVAE":[83.21,0.650],
           "MCOT":[66.42,0.282],"Proposed (Cross-Attn)":[84.46,0.677]}
def baseline_bar(data, title, fname):
    labels = list(data.keys()); acc = [data[k][0] for k in labels]; mcc=[data[k][1]*100 for k in labels]
    x = np.arange(len(labels)); w=0.38
    colors = [RED if "Proposed" in k else GRAY for k in labels]
    fig, ax = plt.subplots(figsize=(5.6,3.6))
    ax.bar(x-w/2, acc, w, color=colors, label="Accuracy")
    ax.bar(x+w/2, mcc, w, color=[c+"88" if False else GRAY_L for c in colors], label="MCC×100")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(title); ax.legend()
    save(fig, fname)
baseline_bar(base_s1, "Stage 1: Comparison with Baselines", "baselines_stage1")
baseline_bar(base_ft, "FakeTT: Comparison with Baselines", "baselines_fakett")

# ---- CV stability ----
cv_s1 = [87.25,88.50,86.75,89.00,87.75]
cv_s2 = [82.50,84.00,81.75,83.50,84.25]
cv_ft = [83.10,84.25,82.70,85.00,84.10]
fig, ax = plt.subplots(figsize=(5.4,3.2))
folds = np.arange(1,6)
for cv, name, col in [(cv_s1,"Stage 1",RED),(cv_s2,"Stage 2",BLUE),(cv_ft,"FakeTT",GREEN)]:
    cv = np.array(cv)
    ax.plot(folds, cv, "o-", color=col, label=f"{name} (μ={cv.mean():.2f})")
ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy (%)"); ax.set_xticks(folds)
ax.set_title("5-Fold Cross-Validation Stability"); ax.legend(fontsize=8)
save(fig, "cv_stability")

print("All bar-chart figures generated.")

saved: fusion_stage1
saved: fusion_stage2
saved: fusion_fakett
saved: ablation_stage1
saved: ablation_stage2
saved: unimodal_stage1
saved: unimodal_stage2
saved: modality_importance_stage1
saved: modality_importance_stage2
saved: baselines_stage1
saved: baselines_fakett
saved: cv_stability
All bar-chart figures generated.


In [39]:
zip_path = "/kaggle/working/journal_figures_600dpi"
shutil.make_archive(zip_path, "zip", FIGDIR)
print("Download ready:", zip_path + ".zip")
print("\nContents:")
for f in sorted(os.listdir(FIGDIR)):
    print(" -", f)

from IPython.display import FileLink
FileLink(zip_path + ".zip")

Download ready: /kaggle/working/journal_figures_600dpi.zip

Contents:
 - ablation_stage1.pdf
 - ablation_stage1.png
 - ablation_stage2.pdf
 - ablation_stage2.png
 - baselines_fakett.pdf
 - baselines_fakett.png
 - baselines_stage1.pdf
 - baselines_stage1.png
 - cm_fakett.pdf
 - cm_fakett.png
 - cm_stage1.pdf
 - cm_stage1.png
 - cm_stage2.pdf
 - cm_stage2.png
 - cv_stability.pdf
 - cv_stability.png
 - fusion_fakett.pdf
 - fusion_fakett.png
 - fusion_stage1.pdf
 - fusion_stage1.png
 - fusion_stage2.pdf
 - fusion_stage2.png
 - modality_importance_stage1.pdf
 - modality_importance_stage1.png
 - modality_importance_stage2.pdf
 - modality_importance_stage2.png
 - roc_fakett.pdf
 - roc_fakett.png
 - roc_stage1.pdf
 - roc_stage1.png
 - unimodal_stage1.pdf
 - unimodal_stage1.png
 - unimodal_stage2.pdf
 - unimodal_stage2.png
 - xai_text_IF.pdf
 - xai_text_IF.png
 - xai_text_PM.pdf
 - xai_text_PM.png
 - xai_text_SC.pdf
 - xai_text_SC.png
 - xai_text_SUS.pdf
 - xai_text_SUS.png
 - xai_visual_IF.pdf

/kaggle/working/journal_figures_600dpi.zip